# xdggs-dggrid4py: A xdggs plugin for IGEO7 DGGS

`xdggs-dggrid4py` aims to provides IGEO7 DGGS indexing for xarray datasets through xdggs. It consists of 2 main functions:

1. Easy access to IGEO7 indexed xarray datasets, including query the datasets with points in WGS84, or zone ID.
2. Convert raster dataset in GeoTiff format to IGEO7 indexed xarray datasets

## Converting a GeoTiff into an IGEO7 DGGS indexed xarray dataset
There are many ways to convert a GeoTiff into a DGGS indexed dataset. The main idea is to assign a pixel with a DGGS zone ID.
- Nearest zone centroid: Assign the pixel to the nearest zone centroid, measured by the CRS supported by the DGGS. For IGEO7 DGGS, it uses WGS84. 
- Zonal statistics: Assign pixels that are within a zone of the DGGS. 
- Weighted average: Overlay the DGGS zone's grid on the raster grid, then, for each zone, calculate the percentage of area overlapped with pixels as the weights. Finally, calculate the zone's value using the weights and raster value.

### General steps of the nearest-neighbourhood method. 

This notebook demonstrates conversion using the nearest neighbourhood method with IGEO7 DGGS.

1. The first step is to determine the refinement level to use. It affects the number of rows of the dataset after conversion.
   - For example, if the refinement level approximately matches the spatial resoultion (in square meters), then the expected number of rows in the resulting dataset should be roughly equal to the number of zones of the region being converted. But if a coarser refinement level is used, the number of rows in the converted result should equal the number of pixels. 

2. Then, generate the zone centroids together with ID that exist in the region of the GeoTiff

3. Assign each pixel to the nearest zone by measuring the distance between the pixel and zone centroids.



### Demonstration 

#### Install the package from `pypi` or install it from GitHub 

In [ ]:
# install it from pypi : 
# !pip install xdggs-dggrid4py
# or install it from GitHub for the lastest commits.
# pip install git+https://github.com/LandscapeGeoinformatics/xdggs-dggrid4py.git

#At the time of preparing this notebook, a bug was found in dggrid4py (https://github.com/allixender/dggrid4py/issues/46) 
# that affect some functions of xdggs (e.g.  `sel_lonlat`). It was fixed in the main branch, but has not yet been built into a release. 
# It needs to install the dggrid4py from git. 
#!pip uninstall -y dggrid4py 
#!pip install git+https://github.com/allixender/dggrid4py.git

#### Import libraries

In [1]:
import xarray as xr
import rioxarray as rio
import os
import warnings
warnings.filterwarnings("ignore")

# xdggs-dggrid4py require both dggrid4py and DGGRID installed
# export the path of DGGRID binary with the environment variable "DGGRID_PATH"
os.environ['DGGRID_PATH']= "/home/dick/micromamba/envs/xdggs/bin/dggrid"

#### Using rioxarray to load a GeoTiff into xarray dataset

In this example, we use a sample collection prepared by the [Landscape and Geoinformatics Lab](https://landscape-geoinformatics.ut.ee/) of the University of Tartu, which is available on Zenodo.

The collection consists of three GeoTiff files (DEM, TWI, Slope) located around Elva, Estonia. [![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.18877265.svg)](https://doi.org/10.5281/zenodo.18877265)

In [2]:
# Using DEM sample to demonstrate the conversion, you can also try with others sample using the following links
# Slope: https://zenodo.org/records/18877265/files/est_topo_slope_10m_clipped.tif?download=1
# TWI: https://zenodo.org/records/18877265/files/est_topo_twi_10m_clipped.tif?download=1

tiff_path = "https://zenodo.org/records/18877265/files/est_topo_dem_10m_clipped.tif?download=1"
raster_ds = rio.open_rasterio(tiff_path, band_as_variable=True, masked=True)
raster_ds

<xarray.Dataset> Size: 8MB
Dimensions:      (x: 1779, y: 1081)
Coordinates:
  * x            (x) float64 14kB 6.334e+05 6.335e+05 ... 6.512e+05 6.512e+05
  * y            (y) float64 9kB 6.468e+06 6.468e+06 ... 6.457e+06 6.457e+06
    spatial_ref  int64 8B 0
Data variables:
    band_1       (y, x) float32 8MB ...
Attributes:
    AREA_OR_POINT:  Area

### Using xdggs-dggrid4py regridding module to convert Raster dataset with the nearest zone centroid method.

`xdggs-dggrid4py` follows the general steps described above for the nearest centroid method, it used S2PointIndex from [pys2Index](https://github.com/benbovy/pys2index) to calculate the distance between pixel coordinates and the zone centroids in WGS84. 

#### Import the conversion method 
  - `xdggs-dggrid4py.regridding.centroid_based.nearestcentroid`: nearest centroid method implementation
  - `xdggs-dggrid4py.regridding.centroid_based.mapblocks_nearestcentroid`: nearest centroid method implementation for parallelism
   
#### Import the regridding method
`xdggs-dggrid4py` supports conversion with parallelism or not.

   - `xdggs_dggrid4py.regridding`
      - `regridding`: perform regridding with a single process  
      - `mapblocks_regridding`: perform regridding with mapblocks parallelism

**Mapblocks parallelism**

To run conversion in parallel, it first partitions the GeoTiff into smaller blocks. Partition is done using xarray chunks; each block is then processed individually, as described in the general steps. Then, it combines the results from each block to form the final dataset to return. The parallelism is achieved using `dask.array.map_blocks`.

**Input parameters for regridding:**
  - `ds`: the xarray dataset to be converted
  - `grid_name`: the name of DGGS to be used for conversion. `IGEO7`
  - `method`: method that used for regridding
  - `coordinates`: the coordinate variables name of the ds in list. Default to `[x, y]`
  - `original_crs`: CRS in wkt format. xdggs-dggrid4py will use the CRS from `crs_wkt` if it exist in the spatial_ref of `ds`. Otherwise, using the wkt supplied. Defualt to `None`. 
  - `refinement_level`: An integer to indicate at which refinement level to convert. Default to `-1`
     - If set to `-1`. It automatically calculates the finest refinement level that matches the spatial resolution. It is done by calculating the surface area of a sphere that represents the GeoTiff region, then dividing that area by the number of pixels to obtain the average area per pixel. The average area per pixel is used to determine the refinement level to be used.
  - `wgs84_geodetic_conversion`: Default to `True`.
       - convert the input clip bounds from WGS84 geodetic coordinates to the authalic sphere coordinates.
       - convert hexagon centroids from the authalic sphere to WGS84 geodetic before assigning pixels to zone centroids.
  - `dggs_vert0_lon`: The longitude of the initial vertex. It is set to 11.20 by default; users can change it to other values by passing the parameter to the function. Default to `11.20`
  - `zone_id_repr`: Which representation is used for the zone ID, it supports : `['int', 'hexstring', 'textual']`, default to 'int'.

In [32]:
%%time

# To perform conversion using nearest centroid mehtod as a single process
refinement_level=11
from xdggs_dggrid4py.regridding.methods.centroid_based import nearestcentroid
from xdggs_dggrid4py.regridding import regridding
igeo7_indexed_ds = regridding(ds=raster_ds,
                              grid_name='igeo7', 
                              method='nearestcentroid',
                              refinement_level=refinement_level,
                              wgs84_geodetic_conversion=True, 
                              zone_id_repr='int')

# To perform conversion using nearest centroid mehtod with mapblocks parallelism
# Try with auto finer refinement level (-1), uncomment below to run
#
#from xdggs_dggrid4py.regridding.methods.centroid_based import mapblocks_nearestcentroid
#from xdggs_dggrid4py.regridding import mapblocks_regridding
#refinement_level=-1
#raster_ds = raster_ds.chunk({'x':200, 'y':200}) # create partitions using chunks
#igeo7_indexed_ds = mapblocks_regridding(ds=raster_ds,
#                                      grid_name='igeo7', 
#                                      method='mapblocks_nearestcentroid',
#                                      refinement_level=refinement_level, 
#                                      wgs84_geodetic_conversion=True, 
#                                      zone_id_repr='int')
#igeo7_indexed_ds

xdggs_dggrid4py.utils area of extent (km^2): 206.60011408860592
xdggs_dggrid4py.utils average area per square grid (km^2): 0.00010743082602019237
CPU times: user 16.2 s, sys: 384 ms, total: 16.5 s
Wall time: 16.9 s


Now, the 2D dataset is converted into a 1D dataset indexed by IGEO7 DGGRS. The related grid info is stored as the `zone_id` attributes.

In [33]:
# Apply mean aggregation to the dataset by zone ID
igeo7_indexed_ds = igeo7_indexed_ds.groupby('zone_id').mean()
igeo7_indexed_ds

<xarray.Dataset> Size: 92kB
Dimensions:      (zone_id: 7664)
Coordinates:
  * zone_id      (zone_id) int64 61kB 18667920867459071 ... 18805349754601471
    spatial_ref  int64 8B 0
Data variables:
    band_1       (zone_id) float32 31kB 73.59 73.35 74.41 ... 68.39 68.5 68.68
Attributes:
    AREA_OR_POINT:  Area

### Optional, create multi-refinement level dataset using IGEO7 z7 hierarchical structure
z7 indexing provides a hierarchical structure between refinement levels, known as the 1 to 7 parent-child hierarchy. Using the zone ID, it is easy to calculate the parent zone at a coarser refinement level it belongs to. Therefore, using this relation, it is simple to perform aggregation at different refinement levels for a multi-refinement level dataset.

In [ ]:
import numpy as np
# function to get the parent zone id of the input zone ID 
def get_parent_z7int(z7_int_zone_id, refinement_level=int):
    binary_repr = np.binary_repr(z7_int_zone_id, width=64)
    binary_repr = binary_repr[:(refinement_level*3)+4]
    binary_repr = binary_repr.ljust(64, '1')
    return int(binary_repr,2)

In [ ]:
# Uisng xarray datatree to store dataset at each refinement level as a group
ds_datatree = xr.DataTree(name="Mutli_refinement_levels_datatree")

import time
import zarr
encode = {}
compressor = zarr.Blosc(cname="zstd", clevel=3, shuffle=2)
for rf_level in range(1, refinement_level+1):
    s = time.time()
    #print(f'working on refinement level {rf_level}')
    tmp = igeo7_indexed_ds.copy()
    tmp['zone_id'] = xr.apply_ufunc(get_parent_z7int, tmp.zone_id, dask='parallelized', vectorize=True,
                                    kwargs={'refinement_level': rf_level})
    tmp = tmp.groupby('zone_id').mean(skipna=True)
    tmp = tmp.chunk('auto')
    enc = {x: {"compressor": compressor} for x in list(tmp.data_vars.keys())}
    enc.update({"zone_id": {"compressor": compressor}})
    ds_datatree = ds_datatree.assign({f"refinement_level_{rf_level}": xr.DataTree(dataset=tmp)})
    encode.update({f"/refinement_level_{rf_level}": enc})
    print(f'done refinement level {rf_level} ({time.time()-s})')
ds_datatree.to_zarr(f'./multi_refinement_level_ds.zarr', encoding=encode)

## Accessing IGEO7 indexed DGGS xarray datasets using `xdggs-dggrid4py`

After the conversion is done, the dataset is indexed with IGEO7 DGGS. However, the index doesn't mean anything to others. Therefore, `xdggs-dggrid4py` provides the `IGEO7Index` as an interface between the user and the IGEO7 DGGS index. It is implemented as a plugin of [xdggs](https://github.com/xarray-contrib/xdggs/tree/main).

In [34]:
# import xdggs and register IGEO7Index by importing the IGEO7Index
import xdggs
from xdggs_dggrid4py import IGEO7Index

In [35]:
# since the xdggs convention uses the name `cell_ids` as the DGGS zone ID, we have to rename the `zone_id` to `cell_ids`
igeo7_indexed_ds = igeo7_indexed_ds.rename({'zone_id': 'cell_ids'})

### Create a IGEO7Index instance for the IGEO7 DGGS dataset
- Invoke the xdggs.decode function to create an IGEO7Index index object for the dataset
- From the output, the `decode` function created an IGEO7Index object using the attributes from `cell_ids` (zone_id)

In [36]:
igeo7_indexed_ds = igeo7_indexed_ds.pipe(xdggs.decode)
igeo7_indexed_ds

<xarray.Dataset> Size: 92kB
Dimensions:      (cell_ids: 7664)
Coordinates:
  * cell_ids     (cell_ids) int64 61kB 18667920867459071 ... 18805349754601471
    spatial_ref  int64 8B 0
Data variables:
    band_1       (cell_ids) float32 31kB 73.59 73.35 74.41 ... 68.39 68.5 68.68
Indexes:
    cell_ids  IGEO7Index(level=11)
Attributes:
    AREA_OR_POINT:  Area

### Using IGEO7Index to access the dataset
- xdggs provides several accessor functions through `<dataset>.dggs`
  - `sel_latlon`:
  - `cell_boundaries`:
  - `explore`:

In [37]:
lats = [58.28459946, 58.27637829, 58.26980135]
lons = [26.41758320, 26.42799669,  26.36551576]
igeo7_indexed_subset = igeo7_indexed_ds.dggs.sel_latlon(lats, lons)
igeo7_indexed_subset

<xarray.Dataset> Size: 44B
Dimensions:      (cell_ids: 3)
Coordinates:
  * cell_ids     (cell_ids) int64 24B 18802036187332607 ... 18801924518182911
    spatial_ref  int64 8B 0
Data variables:
    band_1       (cell_ids) float32 12B 51.27 39.99 83.63
Indexes:
    cell_ids  IGEO7Index(level=11)
Attributes:
    AREA_OR_POINT:  Area

In [38]:
%%time
igeo7_indexed_ds['band_1'].compute().dggs.explore()

CPU times: user 7.55 s, sys: 19.8 ms, total: 7.57 s
Wall time: 7.91 s


### Optional - IGEO7RangeIndexing 
- DGGS datasets can be 
  - `sel_latlon`:
  - `cell_boundaries`:
  - `explore`:

In [39]:
import numpy as np
from xdggs_dggrid4py.zoneid_indexing.igeo7 import IGEO7RangeIndex

In [40]:
# ds_without_index = xr.open_zarr('<zarr path>', create_default_indexes=False, chunks='auto')

# follow from the example from above
# setting the `zoneid_indexing`
igeo7_indexed_ds['cell_ids'].attrs.update({'zoneid_indexing': 'igeo7.RangeIndex'})
igeo7_indexed_ds

<xarray.Dataset> Size: 92kB
Dimensions:      (cell_ids: 7664)
Coordinates:
  * cell_ids     (cell_ids) int64 61kB 18667920867459071 ... 18805349754601471
    spatial_ref  int64 8B 0
Data variables:
    band_1       (cell_ids) float32 31kB 73.59 73.35 74.41 ... 68.39 68.5 68.68
Indexes:
    cell_ids  IGEO7Index(level=11)
Attributes:
    AREA_OR_POINT:  Area

In [41]:
rand_list = np.random.choice(range(0, igeo7_indexed_ds.sizes['cell_ids']),10, replace=False)

In [42]:
%%time
igeo7_rangeindexing_ds = igeo7_indexed_ds.pipe(xdggs.decode)

CPU times: user 23.5 ms, sys: 0 ns, total: 23.5 ms
Wall time: 22.3 ms


In [43]:
%%time
igeo7_rangeindexing_subset = igeo7_rangeindexing_ds.isel({'cell_ids':rand_list}).compute()
igeo7_rangeindexing_subset

CPU times: user 131 ms, sys: 3.83 ms, total: 135 ms
Wall time: 133 ms


<xarray.Dataset> Size: 128B
Dimensions:      (cell_ids: 10)
Coordinates:
  * cell_ids     (cell_ids) uint64 80B 18802110812389375 ... 18802594935734271
    spatial_ref  int64 8B 0
Data variables:
    band_1       (cell_ids) float32 40B 51.87 62.75 68.03 ... 64.72 69.5 69.92
Indexes:
    cell_ids  IGEO7Index(level=11)
Attributes:
    AREA_OR_POINT:  Area

In [55]:
%%time
igeo7_rangeindexing_subset_by_cellids = igeo7_rangeindexing_ds.sel({'cell_ids':igeo7_rangeindexing_subset.cell_ids.data}).compute()
igeo7_rangeindexing_subset_by_cellids

CPU times: user 179 ms, sys: 66 µs, total: 179 ms
Wall time: 180 ms


<xarray.Dataset> Size: 128B
Dimensions:      (cell_ids: 10)
Coordinates:
  * cell_ids     (cell_ids) uint64 80B 18802110812389375 ... 18802594935734271
    spatial_ref  int64 8B 0
Data variables:
    band_1       (cell_ids) float32 40B 51.87 62.75 68.03 ... 64.72 69.5 69.92
Indexes:
    cell_ids  IGEO7Index(level=11)
Attributes:
    AREA_OR_POINT:  Area

In [45]:
# validate the subset selected by RangeIndex is the same with raw ds 
np.isin(igeo7_indexed_ds['band_1'].data[rand_list], igeo7_rangeindexing_subset['band_1'].data)

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True])

In [47]:
np.isin(igeo7_indexed_ds['cell_ids'].data[rand_list], igeo7_rangeindexing_subset['cell_ids'])


array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True])

In [56]:
np.isin(igeo7_indexed_ds['cell_ids'].data[rand_list], igeo7_rangeindexing_subset_by_cellids['cell_ids'])

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True])